In [9]:
import polars as pl

In [ ]:
raw_data = pl.read_csv(
    r"test_path",
    schema_overrides={
        "pending_earn_balance": pl.Float64,
        "pending_redeem_balance": pl.Float64,
        "cleared_balance": pl.Float64,
        "currency_id": pl.Float64,
        "points_earned": pl.Float64,
    },
)

raw_data

zapper_id,contact_number,last_tx_date,last_tx_ref,pending_earn_balance,pending_redeem_balance,cleared_balance,currency_id,points_earned
str,i64,str,str,f64,f64,f64,f64,f64
"""65f0a4dd-f642-4f97-a94e-0439c7…",639988453416,"""2024-01-05 07:55:52.654000+00:…","""Z30905604729""",0.0,0.0,230.0,981.0,0.0
"""bb0c2b3f-adee-46dc-b92b-29d4c9…",639171023436,"""2024-01-05 08:08:44.618000+00:…","""Z32958373874""",0.0,0.0,19.99,981.0,0.0
"""1a717714-41de-4a31-b2b3-8d26d7…",639296458175,"""2024-01-05 08:19:57.174000+00:…","""Z30156177104""",0.0,0.0,740.0,981.0,0.0
"""4699fe97-5521-4a01-afdf-cede30…",639673171002,"""2024-01-05 07:49:52.042000+00:…","""Z34272433060""",0.0,0.0,70.0,981.0,0.0
"""932a39d8-4ce3-4661-97a8-950be0…",639328537161,"""2024-01-05 08:34:27.736000+00:…","""Z32776342596""",0.0,0.0,280.0,981.0,0.0
…,…,…,…,…,…,…,…,…
"""7f2c743f-630d-4f13-a640-896e05…",639918602289,"""2025-10-29 09:43:04.798000+00:…","""Z31983189424""",0.0,0.0,16.5,1050.0,0.0
"""f8db358b-ee96-4dad-a751-3b2531…",639298792397,"""2025-05-06 08:27:32.736000+00:…","""Z30157860336""",0.0,0.0,17.75,1050.0,0.0
"""89d07d63-51f8-4152-9376-3c3dbe…",639761420542,"""2025-08-01 09:54:53.360000+00:…","""Z31122878534""",0.0,0.0,16.75,1050.0,0.0


In [11]:
clean_data = (
    raw_data.with_columns(
        (pl.col("cleared_balance") - pl.col("points_earned")).alias("new_balance")
    )
    .select(["contact_number", "new_balance"])
    .filter(pl.col("new_balance") > 0)
)

clean_data.write_csv("./test_files/2_columns_2.csv", include_header=False)
clean_data


contact_number,new_balance
i64,f64
639988453416,230.0
639171023436,19.99
639296458175,740.0
639673171002,70.0
639328537161,280.0
…,…
639918602289,16.5
639298792397,17.75
639761420542,16.75


In [12]:
chunksize = 1_000
reader = pl.read_csv_batched(r"./test_files/2_columns_2.csv", batch_size=chunksize)

part = 0
while True:
    batches = reader.next_batches(1)  # get up to 1 DataFrame of batch_size
    if not batches:
        break
    for df in batches:
        part += 1
        print(f"Writing part {part}, rows={df.shape[0]}")
        df.write_csv(f"./test_files/big_part_2_{part:03d}.csv", include_header=False)

Writing part 1, rows=1000
Writing part 2, rows=1000
Writing part 3, rows=1000
Writing part 4, rows=1000
Writing part 5, rows=1000
Writing part 6, rows=1000
Writing part 7, rows=1000
Writing part 8, rows=1000
Writing part 9, rows=1000
Writing part 10, rows=562


/var/folders/bj/0krk4gq13b9132z5f9h60ycm0000gn/T/ipykernel_8222/2719638118.py:2: DeprecationWarning: `read_csv_batched` is deprecated; use `scan_csv().collect_batches()` instead.
  reader = pl.read_csv_batched(r"./test_files/2_columns_2.csv", batch_size=chunksize)
